# Complete Framework Validation

This notebook validates the complete mobility-modelling framework from the already constrained demographic-behavioural representation through general opportunity mapping, industry-specific commuting refinement, and final framework assembly.

The previous constrained-profile validation established exhaustive equivalence between the constrained behavioural outputs and the source NTS distributions. This notebook therefore uses that same constrained representation as the behavioural baseline and tests that complete-framework assembly preserves it exactly before adding the opportunity and industry layers.

Validation is deterministic. Every model-relevant complete-framework signature is executed once and mapped back to all constrained profile rows that it represents. Industry is included in a signature only when `Commuting` is retained, because only those outputs can be changed by the industry component.

## 1. Imports and validated inputs

In [1]:
# AI Collaboration Record
# Tool: ChatGPT (OpenAI)
# Purpose: Import the complete-framework functions and load all validated framework inputs.
# Prompt summary: Load the constrained profile space, behavioural evidence, opportunity layer, industry component, POI counts and mapping ontologies required for complete-framework validation.

import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd

TOLERANCE = 1e-8
DOES_NOT_APPLY = "Does not apply"
WORKS_FROM_HOME = "Works mainly from home"
NO_HOUSEHOLD_CAR = "No cars or vans in household"
YOUNG_CHILDREN = {"0-4", "5-11"}

project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

profile_space_path = (
    project_root / "output" / "demographic_profiles"
    / "constrained_representative_profile_space.parquet"
)
behavioural_evidence_path = (
    project_root / "output" / "behavioural_evidence"
    / "nts_behavioural_evidence_layer.json"
)
opportunity_layer_path = (
    project_root / "output" / "opportunity_layer" / "opportunity_layer.json"
)
industry_component_path = (
    project_root / "output" / "opportunity_layer"
    / "industry_to_poi_mapping.json"
)
poi_counts_path = (
    project_root / "data" / "poi" / "processed" / "manchester"
    / "District_manchester_poi_counts.xlsx"
)

required_paths = [
    profile_space_path,
    behavioural_evidence_path,
    opportunity_layer_path,
    industry_component_path,
    poi_counts_path,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required upstream outputs are missing. Re-run the relevant construction "
        "notebooks before this validation:\n" + "\n".join(missing_paths)
    )

representative_profile_space = pd.read_parquet(profile_space_path)
with behavioural_evidence_path.open("r", encoding="utf-8") as file:
    assembled_behavioural_evidence = json.load(file)
with opportunity_layer_path.open("r", encoding="utf-8") as file:
    opportunity_layer = json.load(file)
with industry_component_path.open("r", encoding="utf-8") as file:
    industry_component = json.load(file)
poi_counts = pd.read_excel(poi_counts_path)

from ontology.INDUSTRY_TO_POI_MAPPING import INDUSTRY_TO_POI_MAPPING
from ontology.TRIP_PURPOSE_TO_POI_MAPPING import (
    EXCLUDED_DESTINATION_OUTCOMES,
    TRIP_PURPOSE_TO_POI_MAPPING,
)
from src.complete_framework import construct_complete_framework
from src.demographic_behaviour_layers_integration import (
    DEMOGRAPHIC_ATTRIBUTES,
    construct_constrained_behaviour_representation,
)

print(f"Constrained profiles loaded: {len(representative_profile_space):,}")
print(f"Behavioural purposes loaded: {len(assembled_behavioural_evidence):,}")
print(f"Opportunity mappings loaded: {len(opportunity_layer):,}")
print(f"Industry mappings loaded: {len(industry_component):,}")


Constrained profiles loaded: 2,267,232
Behavioural purposes loaded: 14
Opportunity mappings loaded: 14
Industry mappings loaded: 19


## 2. Independent component-input validation

In [2]:
# AI Collaboration Record
# Tool: ChatGPT (OpenAI)
# Purpose: Define numerical comparison and probability-validation helpers.
# Prompt summary: Create reusable checks for finite non-negative probability distributions and tolerance-based equality of mapping distributions.

def validate_probability_distribution(distribution, name, allow_empty=False):
    if not isinstance(distribution, dict):
        raise TypeError(f"{name} must be a dictionary.")
    if not distribution:
        if allow_empty:
            return
        raise ValueError(f"{name} is empty.")
    values = np.asarray(list(distribution.values()), dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f"{name} contains non-finite values.")
    if (values < 0).any():
        raise ValueError(f"{name} contains negative values.")
    if not np.isclose(values.sum(), 1.0, rtol=0.0, atol=TOLERANCE):
        raise ValueError(f"{name} sums to {values.sum()}, not one.")


def distributions_match(actual, expected, tolerance=TOLERANCE):
    if set(actual) != set(expected):
        return False, math.inf
    categories = sorted(expected)
    actual_values = np.asarray([actual[key] for key in categories], dtype=float)
    expected_values = np.asarray([expected[key] for key in categories], dtype=float)
    differences = np.abs(actual_values - expected_values)
    maximum_difference = float(differences.max()) if len(differences) else 0.0
    return (
        bool(np.allclose(
            actual_values,
            expected_values,
            rtol=0.0,
            atol=tolerance,
            equal_nan=False,
        )),
        maximum_difference,
    )


In [3]:
# AI Collaboration Record
# Tool: ChatGPT (OpenAI)
# Purpose: Reconstruct and validate the general opportunity and industry components against their source inputs.
# Prompt summary: Independently recompute log-transformed POI distributions from the mapping ontologies and compare every exported general-opportunity and industry distribution with the reference values.

required_profile_columns = set(DEMOGRAPHIC_ATTRIBUTES)
assert required_profile_columns.issubset(representative_profile_space.columns)
assert not representative_profile_space[list(DEMOGRAPHIC_ATTRIBUTES)].isna().any().any()
assert not representative_profile_space.duplicated(
    subset=list(DEMOGRAPHIC_ATTRIBUTES)
).any()

poi_counts_reference = poi_counts.copy()
poi_counts_reference["Log Count"] = np.log1p(
    poi_counts_reference["POI Count"]
)
log_count_lookup = poi_counts_reference.set_index("POI Category")["Log Count"]

reference_opportunity_layer = {}
for _, mapping_row in TRIP_PURPOSE_TO_POI_MAPPING.iterrows():
    purpose = mapping_row["Shared Trip Purpose"]
    strategy = mapping_row["Mapping Strategy"]
    categories = mapping_row["POI Categories"]

    if strategy == "Excluded Mapping":
        distribution = {}
    elif strategy in {"Direct Mapping", "Representative Mapping"}:
        distribution = {categories[0]: 1.0}
    elif strategy == "Context-dependent Mapping":
        selected_counts = log_count_lookup.loc[categories]
        distribution = {
            category: float(value / selected_counts.sum())
            for category, value in selected_counts.items()
        }
    else:
        raise ValueError(f"Unknown mapping strategy: {strategy}")

    reference_opportunity_layer[purpose] = {
        "mapping_strategy": strategy,
        "poi_distribution": distribution,
    }
    if purpose in EXCLUDED_DESTINATION_OUTCOMES:
        reference_opportunity_layer[purpose]["allocation_outcome"] = (
            EXCLUDED_DESTINATION_OUTCOMES[purpose]
        )

reference_industry_component = {}
for industry, target in INDUSTRY_TO_POI_MAPPING.items():
    if target is None:
        reference_industry_component[industry] = None
    elif isinstance(target, list):
        selected_counts = log_count_lookup.loc[target]
        reference_industry_component[industry] = {
            category: float(value / selected_counts.sum())
            for category, value in selected_counts.items()
        }
    else:
        reference_industry_component[industry] = {target: 1.0}

assert set(opportunity_layer) == set(reference_opportunity_layer)
assert set(industry_component) == set(reference_industry_component)

component_validation_records = []
for purpose, reference in reference_opportunity_layer.items():
    actual = opportunity_layer[purpose]
    distribution_passed, maximum_difference = distributions_match(
        actual["poi_distribution"], reference["poi_distribution"]
    )
    strategy_passed = actual["mapping_strategy"] == reference["mapping_strategy"]
    excluded_outcome_passed = (
        reference["mapping_strategy"] != "Excluded Mapping"
        or actual.get("allocation_outcome") == reference.get("allocation_outcome")
    )
    if reference["mapping_strategy"] == "Excluded Mapping":
        validate_probability_distribution(
            actual["poi_distribution"], f"Opportunity {purpose}", allow_empty=True
        )
    else:
        validate_probability_distribution(
            actual["poi_distribution"], f"Opportunity {purpose}"
        )
    component_validation_records.append({
        "Component": "General opportunity",
        "Mapping": purpose,
        "Maximum Difference": maximum_difference,
        "Result": "PASS" if (
            distribution_passed and strategy_passed and excluded_outcome_passed
        ) else "FAIL",
    })

for industry, reference in reference_industry_component.items():
    actual = industry_component[industry]
    if reference is None:
        distribution_passed = actual is None
        maximum_difference = 0.0 if distribution_passed else math.inf
    else:
        validate_probability_distribution(actual, f"Industry {industry}")
        distribution_passed, maximum_difference = distributions_match(actual, reference)
    component_validation_records.append({
        "Component": "Industry",
        "Mapping": industry,
        "Maximum Difference": maximum_difference,
        "Result": "PASS" if distribution_passed else "FAIL",
    })

component_validation_summary = pd.DataFrame(component_validation_records)
assert component_validation_summary["Result"].eq("PASS").all(), (
    "One or more exported opportunity/industry mappings differ from their "
    "independently reconstructed source values."
)

print(component_validation_summary.groupby("Component").agg(
    Mappings_Validated=("Mapping", "size"),
    Maximum_Difference=("Maximum Difference", "max"),
    Failed=("Result", lambda values: values.eq("FAIL").sum()),
).to_string())
print("\nPASS: Exported opportunity and industry components match source inputs.")


                     Mappings_Validated  Maximum_Difference  Failed
Component                                                          
General opportunity                  14                 0.0       0
Industry                             19                 0.0       0

PASS: Exported opportunity and industry components match source inputs.


## 3. Complete-framework signature coverage

In [4]:
# AI Collaboration Record
# Tool: ChatGPT (OpenAI)
# Purpose: Partition the constrained profile space and derive exact complete-framework signatures.
# Prompt summary: Assign every constrained row to exactly one behavioural case; retain Industry in the cache key only for workplace profiles whose Commuting distribution is refined.

profiles = representative_profile_space
has_household_car = profiles["Car Ownership"].ne(NO_HOUSEHOLD_CAR)
is_young_child = profiles["Age"].isin(YOUNG_CHILDREN)
industry_applies = profiles["Industry"].ne(DOES_NOT_APPLY)
works_from_home = profiles["Distance travelled to work"].eq(WORKS_FROM_HOME)
work_distance_not_applicable = profiles["Distance travelled to work"].eq(
    DOES_NOT_APPLY
)
workplace_based = industry_applies & ~works_from_home & ~work_distance_not_applicable

case_definitions = [
    (profiles["Age"].eq("0-4") & has_household_car,
     "Child aged 0-4 with household car"),
    (profiles["Age"].eq("0-4") & ~has_household_car,
     "Child aged 0-4 without household car"),
    (profiles["Age"].eq("5-11") & has_household_car,
     "Child aged 5-11 with household car"),
    (profiles["Age"].eq("5-11") & ~has_household_car,
     "Child aged 5-11 without household car"),
    (~is_young_child & ~industry_applies & work_distance_not_applicable
     & has_household_car,
     "Industry-not-applicable profile aged 12+ with household car"),
    (~is_young_child & ~industry_applies & work_distance_not_applicable
     & ~has_household_car,
     "Industry-not-applicable profile aged 12+ without household car"),
    (~is_young_child & industry_applies & works_from_home & has_household_car,
     "Home-working profile with household car"),
    (~is_young_child & industry_applies & works_from_home & ~has_household_car,
     "Home-working profile without household car"),
    (~is_young_child & workplace_based & has_household_car,
     "Workplace-based profile with household car"),
    (~is_young_child & workplace_based & ~has_household_car,
     "Workplace-based profile without household car"),
]

case_membership_count = sum(mask.astype("int8") for mask, _ in case_definitions)
assert case_membership_count.eq(1).all(), (
    "The behavioural cases do not partition every constrained row exactly once."
)

case_assignment = pd.Series(pd.NA, index=profiles.index, dtype="object")
for mask, case_name in case_definitions:
    case_assignment.loc[mask] = case_name
assert case_assignment.notna().all()

commuting_retained = workplace_based
effective_industry = profiles["Industry"].where(
    commuting_retained, "Not applicable to final output"
)

signature_key_columns = ["Case", "Age", "Gender", "Effective Industry"]
signature_membership_frame = pd.DataFrame({
    "Case": case_assignment,
    "Age": profiles["Age"].to_numpy(),
    "Gender": profiles["Gender"].to_numpy(),
    "Effective Industry": effective_industry.to_numpy(),
    "Commuting Retained": commuting_retained.to_numpy(),
    "Profile Row Position": np.arange(len(profiles), dtype=np.int64),
})

signature_population_counts = (
    signature_membership_frame
    .groupby(signature_key_columns, observed=True, sort=False)
    .size()
    .rename("Rows Represented")
    .reset_index()
)
signature_representatives = (
    signature_membership_frame
    .drop_duplicates(signature_key_columns, keep="first")
    [signature_key_columns + ["Commuting Retained", "Profile Row Position"]]
)
signature_population_summary = (
    signature_population_counts
    .merge(
        signature_representatives,
        on=signature_key_columns,
        how="left",
        validate="one_to_one",
    )
    .sort_values(signature_key_columns)
    .reset_index(drop=True)
)

assert int(signature_population_summary["Rows Represented"].sum()) == len(profiles)
assert not signature_population_summary.duplicated(signature_key_columns).any()

coverage_by_case = (
    signature_population_summary.groupby("Case", sort=False).agg(
        Unique_Complete_Framework_Signatures=("Rows Represented", "size"),
        Constrained_Rows_Covered=("Rows Represented", "sum"),
    )
)
print(coverage_by_case.to_string())
print(
    f"\nComplete constrained-row coverage: {len(profiles):,} rows mapped to "
    f"{len(signature_population_summary):,} exact complete-framework signatures."
)


                                                                Unique_Complete_Framework_Signatures  Constrained_Rows_Covered
Case                                                                                                                          
Child aged 0-4 with household car                                                                  2                       570
Child aged 0-4 without household car                                                               2                       190
Child aged 5-11 with household car                                                                 2                       570
Child aged 5-11 without household car                                                              2                       190
Home-working profile with household car                                                           14                    238032
Home-working profile without household car                                                        14           

## 4. Execute and validate every complete-framework signature

In [5]:
# AI Collaboration Record
# Tool: ChatGPT (OpenAI)
# Purpose: Define layer-preservation and final-output validation helpers.
# Prompt summary: Compare the behavioural baseline with the complete-framework input state, validate opportunity/industry placement, and check the returned presentation structures.

def compare_behavioural_core(baseline, complete_profile):
    baseline_behaviour = baseline["behavioural_representation"]
    complete_behaviour = complete_profile["behavioural_representation"]

    trip_match, trip_max_difference = distributions_match(
        complete_behaviour["trip_purpose_distribution"],
        baseline_behaviour["trip_purpose_distribution"],
    )
    baseline_conditioned = baseline_behaviour["purpose_conditioned_distribution"]
    complete_conditioned = complete_behaviour["purpose_conditioned_distribution"]
    if set(baseline_conditioned) != set(complete_conditioned):
        return False, math.inf

    maximum_difference = trip_max_difference
    for purpose in baseline_conditioned:
        baseline_details = baseline_conditioned[purpose]
        complete_details = complete_conditioned[purpose]
        if not np.isclose(
            complete_details["trip_purpose_probability"],
            baseline_details["trip_purpose_probability"],
            rtol=0.0,
            atol=TOLERANCE,
        ):
            return False, math.inf
        for key in ["temporal_distribution", "travel_mode_distribution"]:
            passed, difference = distributions_match(
                complete_details[key], baseline_details[key]
            )
            if not passed:
                return False, difference
            maximum_difference = max(maximum_difference, difference)
    return trip_match, maximum_difference


def expected_removed_purposes(profile):
    if profile["Industry"] == DOES_NOT_APPLY:
        return {"Commuting", "Business"}
    if profile["Distance travelled to work"] == WORKS_FROM_HOME:
        return {"Commuting"}
    return set()


def expected_removed_modes(profile):
    removals = set()
    if profile["Age"] in YOUNG_CHILDREN:
        removals.update({"Car or van driver", "Motorcycle"})
    if profile["Car Ownership"] == NO_HOUSEHOLD_CAR:
        removals.update({"Car or van driver", "Car or van passenger"})
    return removals


def validate_framework_dataframes(frames, retained_purposes):
    expected_keys = {
        "demographic_layer",
        "trip_purpose_probabilities",
        "temporal_distributions",
        "travel_mode_distributions",
        "opportunity_layer",
    }
    if set(frames) != expected_keys:
        return False
    if len(frames["demographic_layer"]) != 1:
        return False
    if not set(DEMOGRAPHIC_ATTRIBUTES).issubset(frames["demographic_layer"].columns):
        return False

    purpose_frame = frames["trip_purpose_probabilities"]
    if set(purpose_frame["Trip Purpose"]) != retained_purposes:
        return False
    probabilities = pd.to_numeric(purpose_frame["Probability"], errors="coerce")
    if probabilities.isna().any() or (probabilities < 0).any():
        return False

    for frame_name in ["temporal_distributions", "travel_mode_distributions"]:
        frame = frames[frame_name]
        if "Trip Purpose" not in frame or set(frame["Trip Purpose"]) != retained_purposes:
            return False

    opportunity_frame = frames["opportunity_layer"]
    if set(opportunity_frame["Trip Purpose"]) != retained_purposes:
        return False
    if opportunity_frame[["Trip Purpose", "POI Category"]].isna().any().any():
        return False
    return True


In [6]:
# AI Collaboration Record
# Tool: ChatGPT (OpenAI)
# Purpose: Execute every exact complete-framework signature and record all layer-level checks.
# Prompt summary: For each representative signature, compare the complete-framework behavioural core with the constrained baseline, validate retained/excluded opportunity mappings, verify conditional industry refinement and check final output structure.

signature_validation_records = []

for _, signature in signature_population_summary.iterrows():
    row_position = int(signature["Profile Row Position"])
    rows_represented = int(signature["Rows Represented"])
    profile = profiles.iloc[row_position].copy()
    commuting_should_be_retained = bool(signature["Commuting Retained"])

    record = {
        "Case": signature["Case"],
        "Age": signature["Age"],
        "Gender": signature["Gender"],
        "Effective Industry": signature["Effective Industry"],
        "Rows Represented": rows_represented,
        "Commuting Retained": commuting_should_be_retained,
    }

    try:
        baseline = construct_constrained_behaviour_representation(
            representative_profile=profile,
            assembled_behavioural_evidence=assembled_behavioural_evidence,
        )
        complete_profile, framework_dataframes = construct_complete_framework(
            representative_profile=profile,
            assembled_behavioural_evidence=assembled_behavioural_evidence,
            opportunity_layer=opportunity_layer,
            industry_component=industry_component,
        )

        baseline_match, behavioural_maximum_difference = compare_behavioural_core(
            baseline, complete_profile
        )
        demographic_match = (
            complete_profile["demographic_profile"]
            == baseline["demographic_profile"]
        )

        conditioned = complete_profile["behavioural_representation"][
            "purpose_conditioned_distribution"
        ]
        retained_purposes = set(conditioned)
        removed_purposes = expected_removed_purposes(profile)
        expected_purposes = set(assembled_behavioural_evidence) - removed_purposes
        retained_purpose_set_correct = retained_purposes == expected_purposes

        mode_removals = expected_removed_modes(profile)
        mode_removals_correct = all(
            mode_removals.isdisjoint(
                details["travel_mode_distribution"]
            )
            for details in conditioned.values()
        )

        all_behavioural_distributions_valid = True
        for purpose, details in conditioned.items():
            validate_probability_distribution(
                details["temporal_distribution"], f"{purpose} temporal"
            )
            validate_probability_distribution(
                details["travel_mode_distribution"], f"{purpose} travel mode"
            )
        validate_probability_distribution(
            complete_profile["behavioural_representation"][
                "trip_purpose_distribution"
            ],
            "Complete-framework trip purpose",
        )

        opportunity_mapping_correct = True
        excluded_mapping_correct = True
        industry_refinement_correct = True
        opportunity_maximum_difference = 0.0
        industry_maximum_difference = 0.0

        for purpose, details in conditioned.items():
            source_mapping = opportunity_layer[purpose]
            opportunity_mapping_correct &= (
                details["mapping_strategy"] == source_mapping["mapping_strategy"]
            )

            if source_mapping["mapping_strategy"] == "Excluded Mapping":
                excluded_mapping_correct &= (
                    details["poi_distribution"] == {}
                    and details.get("allocation_outcome")
                    == source_mapping["allocation_outcome"]
                )
                industry_refinement_correct &= (
                    details.get("industry_refinement_applied") is False
                    and "industry" not in details
                    and "general_poi_distribution" not in details
                )
                validate_probability_distribution(
                    details["poi_distribution"],
                    f"{purpose} excluded opportunity",
                    allow_empty=True,
                )
            elif purpose == "Commuting":
                expected_general = source_mapping["poi_distribution"]
                general_passed, general_difference = distributions_match(
                    details.get("general_poi_distribution", {}), expected_general
                )
                expected_industry = industry_component[profile["Industry"]]
                final_passed, final_difference = distributions_match(
                    details["poi_distribution"], expected_industry
                )
                industry_refinement_correct &= (
                    commuting_should_be_retained
                    and details.get("industry_refinement_applied") is True
                    and details.get("industry") == profile["Industry"]
                    and general_passed
                    and final_passed
                )
                industry_maximum_difference = max(
                    industry_maximum_difference,
                    general_difference,
                    final_difference,
                )
                validate_probability_distribution(
                    details["poi_distribution"], f"{purpose} industry opportunity"
                )
            else:
                distribution_passed, maximum_difference = distributions_match(
                    details["poi_distribution"], source_mapping["poi_distribution"]
                )
                opportunity_mapping_correct &= distribution_passed
                opportunity_maximum_difference = max(
                    opportunity_maximum_difference, maximum_difference
                )
                industry_refinement_correct &= (
                    details.get("industry_refinement_applied") is False
                    and "industry" not in details
                    and "general_poi_distribution" not in details
                )
                validate_probability_distribution(
                    details["poi_distribution"], f"{purpose} opportunity"
                )

        commuting_present = "Commuting" in conditioned
        commuting_presence_correct = (
            commuting_present == commuting_should_be_retained
        )
        if not commuting_should_be_retained:
            industry_refinement_correct &= all(
                details.get("industry_refinement_applied") is False
                and "industry" not in details
                and "general_poi_distribution" not in details
                for details in conditioned.values()
            )

        final_output_structure_valid = validate_framework_dataframes(
            framework_dataframes, retained_purposes
        )

        signature_passed = all([
            baseline_match,
            demographic_match,
            retained_purpose_set_correct,
            mode_removals_correct,
            all_behavioural_distributions_valid,
            opportunity_mapping_correct,
            excluded_mapping_correct,
            commuting_presence_correct,
            industry_refinement_correct,
            final_output_structure_valid,
        ])

        record.update({
            "Behavioural Baseline Preserved": baseline_match,
            "Behavioural Maximum Difference": behavioural_maximum_difference,
            "Demographic Profile Preserved": demographic_match,
            "Retained Purpose Set Correct": retained_purpose_set_correct,
            "Mode Removals Preserved": mode_removals_correct,
            "Behavioural Distributions Valid": all_behavioural_distributions_valid,
            "Opportunity Mapping Correct": opportunity_mapping_correct,
            "Opportunity Maximum Difference": opportunity_maximum_difference,
            "Excluded Mapping Correct": excluded_mapping_correct,
            "Commuting Presence Correct": commuting_presence_correct,
            "Industry Refinement Correct": industry_refinement_correct,
            "Industry Maximum Difference": industry_maximum_difference,
            "Final Output Structure Valid": final_output_structure_valid,
            "Retained Purposes Tested": len(retained_purposes),
            "Result": "PASS" if signature_passed else "FAIL",
            "Error": "",
        })
    except Exception as error:
        record.update({
            "Behavioural Baseline Preserved": False,
            "Behavioural Maximum Difference": np.nan,
            "Demographic Profile Preserved": False,
            "Retained Purpose Set Correct": False,
            "Mode Removals Preserved": False,
            "Behavioural Distributions Valid": False,
            "Opportunity Mapping Correct": False,
            "Opportunity Maximum Difference": np.nan,
            "Excluded Mapping Correct": False,
            "Commuting Presence Correct": False,
            "Industry Refinement Correct": False,
            "Industry Maximum Difference": np.nan,
            "Final Output Structure Valid": False,
            "Retained Purposes Tested": 0,
            "Result": "FAIL",
            "Error": str(error),
        })

    signature_validation_records.append(record)

signature_validation_details = pd.DataFrame(signature_validation_records)
failed_signature_details = signature_validation_details.loc[
    signature_validation_details["Result"].eq("FAIL")
]

validation_by_case = (
    signature_validation_details.groupby("Case", sort=False).agg(
        Signatures_Executed=("Result", "size"),
        Profile_Rows_Covered=("Rows Represented", "sum"),
        Passed=("Result", lambda values: values.eq("PASS").sum()),
        Failed=("Result", lambda values: values.eq("FAIL").sum()),
        Opportunity_Mappings_Tested=("Retained Purposes Tested", "sum"),
        Behavioural_Maximum_Difference=("Behavioural Maximum Difference", "max"),
        Opportunity_Maximum_Difference=("Opportunity Maximum Difference", "max"),
        Industry_Maximum_Difference=("Industry Maximum Difference", "max"),
    )
)
print(validation_by_case.to_string())
if not failed_signature_details.empty:
    print("\nFailed signature details:\n")
    print(failed_signature_details.to_string(index=False))


                                                                Signatures_Executed  Profile_Rows_Covered  Passed  Failed  Opportunity_Mappings_Tested  Behavioural_Maximum_Difference  Opportunity_Maximum_Difference  Industry_Maximum_Difference
Case                                                                                                                                                                                                                                               
Child aged 0-4 with household car                                                 2                   570       2       0                           24                             0.0                             0.0                          0.0
Child aged 0-4 without household car                                              2                   190       2       0                           24                             0.0                             0.0                          0.0
Child aged 5-11 with hou

## 5. Exhaustive coverage assertions and final result

In [7]:
# AI Collaboration Record
# Tool: ChatGPT (OpenAI)
# Purpose: Assert complete passed-signature coverage and report the final validation evidence.
# Prompt summary: Fail unless every exact complete-framework signature passes and all constrained rows, retained purposes and applicable industry refinements are covered.

total_profiles = len(profiles)
total_signatures = len(signature_validation_details)
passed_signatures = int(signature_validation_details["Result"].eq("PASS").sum())
failed_signatures = total_signatures - passed_signatures
rows_covered = int(signature_validation_details["Rows Represented"].sum())
rows_covered_by_passed_signatures = int(
    signature_validation_details.loc[
        signature_validation_details["Result"].eq("PASS"), "Rows Represented"
    ].sum()
)
rows_mapped_to_failed_signatures = total_profiles - rows_covered_by_passed_signatures
industry_signatures_executed = int(
    signature_validation_details["Commuting Retained"].sum()
)
non_industry_signatures_executed = total_signatures - industry_signatures_executed
signature_purpose_comparisons = int(
    signature_validation_details["Retained Purposes Tested"].sum()
)
row_level_purpose_outcomes_covered = int((
    signature_validation_details["Rows Represented"]
    * signature_validation_details["Retained Purposes Tested"]
).sum())

assert rows_covered == total_profiles
assert failed_signatures == 0, (
    "One or more complete-framework signatures failed:\n"
    + failed_signature_details.to_string(index=False)
)
assert rows_mapped_to_failed_signatures == 0
assert signature_validation_details["Behavioural Baseline Preserved"].all()
assert signature_validation_details["Demographic Profile Preserved"].all()
assert signature_validation_details["Retained Purpose Set Correct"].all()
assert signature_validation_details["Mode Removals Preserved"].all()
assert signature_validation_details["Behavioural Distributions Valid"].all()
assert signature_validation_details["Opportunity Mapping Correct"].all()
assert signature_validation_details["Excluded Mapping Correct"].all()
assert signature_validation_details["Commuting Presence Correct"].all()
assert signature_validation_details["Industry Refinement Correct"].all()
assert signature_validation_details["Final Output Structure Valid"].all()

overall_validation_summary = pd.DataFrame({
    "Measure": [
        "Constrained profiles covered",
        "Unique complete-framework signatures executed",
        "Framework executions avoided through exact signatures",
        "Non-industry signatures executed",
        "Industry-refined commuting signatures executed",
        "General opportunity mappings independently validated",
        "Industry mappings independently validated",
        "Signature-level retained-purpose mappings tested",
        "Row-level retained-purpose outcomes covered",
        "Constrained rows covered by passed signatures",
        "Constrained rows mapped to failed signatures",
        "Behavioural maximum difference",
        "Opportunity maximum difference",
        "Industry maximum difference",
        "Overall result",
    ],
    "Result": [
        total_profiles,
        total_signatures,
        total_profiles - total_signatures,
        non_industry_signatures_executed,
        industry_signatures_executed,
        int((component_validation_summary["Component"] == "General opportunity").sum()),
        int((component_validation_summary["Component"] == "Industry").sum()),
        signature_purpose_comparisons,
        row_level_purpose_outcomes_covered,
        rows_covered_by_passed_signatures,
        rows_mapped_to_failed_signatures,
        "{:.3e}".format(
            signature_validation_details["Behavioural Maximum Difference"].max()
        ),
        "{:.3e}".format(
            signature_validation_details["Opportunity Maximum Difference"].max()
        ),
        "{:.3e}".format(
            signature_validation_details["Industry Maximum Difference"].max()
        ),
        "PASS",
    ],
})

print(overall_validation_summary.to_string(index=False))
print(
    "\nPASS: Every constrained profile is covered through an exact "
    "complete-framework signature. The constrained behavioural baseline was "
    "preserved, all retained purposes received the expected opportunity mapping, "
    "industry refinement was applied only to retained Commuting, and every final "
    "framework output passed the structural and probability checks."
)


                                              Measure    Result
                         Constrained profiles covered   2267232
        Unique complete-framework signatures executed       568
Framework executions avoided through exact signatures   2266664
                     Non-industry signatures executed        64
       Industry-refined commuting signatures executed       504
 General opportunity mappings independently validated        14
            Industry mappings independently validated        19
     Signature-level retained-purpose mappings tested      7852
          Row-level retained-purpose outcomes covered  31332672
        Constrained rows covered by passed signatures   2267232
         Constrained rows mapped to failed signatures         0
                       Behavioural maximum difference 0.000e+00
                       Opportunity maximum difference 0.000e+00
                          Industry maximum difference 0.000e+00
                                       O

## Interpretation

Passing this notebook supports the following precise statement:

> All constrained profiles were covered through exact complete-framework signatures. Across those signatures, the previously validated demographic-behavioural representation was preserved; general opportunity mappings, excluded-purpose handling, and industry-specific commuting refinement matched their independently reconstructed source components; and all final framework outputs were structurally valid.

The framework is executed once per unique model-relevant signature rather than once per original row. Therefore, the evidence demonstrates exhaustive row coverage through exact signatures, not millions of separate framework executions.